## ### **Gold** Layer Incremental
Incremental Gold processing plus latest and timestamped Volume snapshots.

### ## **Step 1 - Import and setup**

This cell imports required helpers, switches to the right calalog, makes sure the Gold SChema exists and creates

- **a Gold_run_id**
- a Run date string
- a run timestamp string

These are used for tracking and snapshot publishing.

In [0]:
# Import PySpark SQL functions for DataFrame operations
from pyspark.sql import functions as F

# Import DeltaTable for Delta Lake operations (merge/upsert)
from delta.tables import DeltaTable

# Import datetime for handling timestamps
from datetime import datetime

# Import uuid for generating unique run IDs
import uuid

In [0]:
# Switch to the databricks_dev catalog
spark.sql("use catalog databricks_dev")

# Create the 02_gold schema if it does not exist
spark.sql("create schema if not exists 02_gold")

# Generate a unique Gold run ID for this processing run
gold_run_id = str(uuid.uuid4())

# Get the current UTC timestamp as a string for folder naming or logging
run_ts_str = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

# Get the current UTC date as a string for partitioning or logging
run_date_str = datetime.utcnow().strftime("%Y-%m-%d")

# Print the current Gold run ID
print("current gold run id", gold_run_id)

# Print the run timestamp string
print("run timestamp folder", run_ts_str)

### **Step 2 - Create Gold control table**
This table stores the latest gold processing state.

it tells gold:

- which silver data was processed last time
- how many Gold rows were merged in the last run

In [0]:
# Create the Gold processing_control table if it does not exist
# This table tracks the state of Gold processing, including:
# - Which silver data was processed last time
# - How many Gold rows were merged in the last run
# - Run status and metadata for auditing
spark.sql("""
    create table if not exists databricks_dev.`02_gold`.processing_control(
        layer string,                              -- Data layer (e.g., 'gold')
        entity_name string,                        -- Name of the entity processed
        last_processed_silver_run_id string,        -- Last processed Silver run ID
        last_processed_silver_run_ts timestamp,     -- Last processed Silver run timestamp
        rows_merged bigint,                        -- Number of rows merged in last run
        run_status string,                         -- Status of the run (e.g., 'Success')
        gold_run_id string,                        -- Unique Gold run ID
        updated_at timestamp                       -- Timestamp of last update
    )
    using delta
""")

###  **Step 3 - Helper functions**
  This cell defines reusbale Gold functions:

-     upsert_to_gold() merges data into Gold current-state tables
-    get_last_processed_silver_ts() reads the Gold watermark from the control table
-     upsert_gold_control() uodates Gold control after successful run

In [0]:
def upsert_to_gold(df_source, target_table, join_key):
    # Check if the target Gold table exists
    if spark.catalog.tableExists(target_table):
        # If the table exists, perform a Delta Lake merge (upsert)
        dt = DeltaTable.forName(spark, target_table)
        (
            dt.alias("t")
            .merge(
                df_source.alias("s"),
                "t." + join_key + " = s." + join_key,  # Join on the specified key
            )
            .whenMatchedUpdateAll()  # Update all columns if matched
            .whenNotMatchedInsertAll()  # Insert all columns if not matched
            .execute()
        )
    else:
        # If the table does not exist, create it using the source DataFrame
        df_source.write.format("delta").saveAsTable(target_table)

In [0]:
def get_last_processed_silver_ts(entity_name:str):
    # Query the Gold processing_control table for the latest successful run for the given entity
    ctrl = (
        spark.table("databricks_dev.`02_gold`.processing_control")
        .filter(
            (F.col("layer") == "gold") &
            (F.col("entity_name") == entity_name) &
            (F.col("run_status") == "Success")
        )
        .orderBy(F.col("updated_at").desc())  # Order by most recent update
        .limit(1)  # Get the latest record
        .collect()  # Collect results to driver
    )
    rows = ctrl  # Assign collected rows to variable

    # If no rows found, return None (no previous successful run)
    if not rows:
        return None
    else:
        # Return the last processed silver run timestamp from the latest control record
        return rows[0]["last_processed_silver_run_ts"]

In [0]:
def upsert_gold_control(entity_name, last_processed_silver_run_id, last_processed_silver_run_ts, rows_merged):
    # Create a DataFrame with the latest Gold control metadata for this run
    ctrl_df = spark.createDataFrame(
        [
            (
                "gold",  # Data layer
                entity_name,  # Name of the entity processed
                last_processed_silver_run_id,  # Last processed Silver run ID
                last_processed_silver_run_ts,  # Last processed Silver run timestamp
                int(rows_merged),  # Number of rows merged in this run
                "Success",  # Run status
                gold_run_id,  # Unique Gold run ID
                datetime.utcnow()  # Timestamp of update
            )
        ],
        schema="""
            layer string,
            entity_name string,
            last_processed_silver_run_id string,
            last_processed_silver_run_ts timestamp,
            rows_merged bigint,
            run_status string,
            gold_run_id string,
            updated_at timestamp
        """
    )
    # Get a DeltaTable reference for the Gold processing_control table
    dt = DeltaTable.forName(spark, "databricks_dev.`02_gold`.processing_control")
    # Merge the new control record into the processing_control table
    (
        dt.alias("t")
        .merge(
            ctrl_df.alias("s"),
            "t.entity_name = s.entity_name",  # Match on entity_name
        )
        .whenMatchedUpdate(
            set={
                "last_processed_silver_run_id": "s.last_processed_silver_run_id",
                "last_processed_silver_run_ts": "s.last_processed_silver_run_ts",
                "rows_merged": "s.rows_merged",
                "run_status": "s.run_status",
                "gold_run_id": "s.gold_run_id",
                "updated_at": "s.updated_at"
            }
        )  # Update fields if matched
        .whenNotMatchedInsertAll()  # Insert new record if not matched
        .execute()
    )

###  **Step 4 - Read changed silver rows only**

This cell reads the full silver current state tables but filters out only the rows that changed since the last Gold run,

This is the starting Point for Gold incremental processing.

In [0]:
# Step 4 - Read changed silver rows only
# This cell reads the full silver current state tables but filters out only the rows that changed since the last Gold run.
# This is the starting point for Gold incremental processing.

# Import required functions
from pyspark.sql import functions as F

# Get the last processed silver run timestamp from the Gold control table for 'orders_information'
last_gold_row = (
    spark.table("databricks_dev.`02_gold`.processing_control")
    .filter(
        (F.col("layer") == "gold")
        & (F.col("entity_name") == "orders_information")
        & (F.col("run_status") == "Success")
    )
    .orderBy(F.col("updated_at").desc())
    .limit(1)
    .collect()
)
last_gold_ts = last_gold_row[0]["last_processed_silver_run_ts"] if last_gold_row else None
print(" last gold processed silver timestamp for gold= ", last_gold_ts)

# Read current state Silver tables for orders, products, and payments
silver_orders_current = spark.read.table("databricks_dev.`01_silver`.orders_transformed")
silver_product_current = spark.read.table("databricks_dev.`01_silver`.product_transformed")
silver_payment_current = spark.read.table("databricks_dev.`01_silver`.payments_transformed")
 
# If this is the first Gold run, consider all rows as changed; otherwise, filter for rows updated since last_gold_ts
if last_gold_ts is None:
    changed_orders = silver_orders_current
    changed_products = silver_product_current
    changed_payments = silver_payment_current
else:
    changed_orders = silver_orders_current.filter(F.col("bronze_ingested_ts") > F.lit(last_gold_ts))
    changed_products = silver_product_current.filter(F.col("bronze_ingested_ts") > F.lit(last_gold_ts))
    changed_payments = silver_payment_current.filter(F.col("bronze_ingested_ts") > F.lit(last_gold_ts))

# Count the number of changed rows in each entity
changed_orders_count = changed_orders.count()
changed_products_count = changed_products.count()
changed_payments_count = changed_payments.count()

print("changed_orders_count= ", changed_orders_count)
print("changed_products_count= ", changed_products_count)
print("changed_payments_count= ", changed_payments_count)

### **Step 5 - Find impacted order IDs**
Gold is bulit at order grain, so if anything changes in orders, products or payments, we identify which **order_id** values are impacted.

Only those order IDS are rebuilt in gold

In [0]:
# Select distinct order IDs from changed orders
impacted_from_orders = changed_orders.select("order_id").distinct()

# Select distinct order IDs from changed payments
impacted_from_payments = changed_payments.select("order_id").distinct()

# Find order IDs impacted by changed products by joining with current orders on product_id
impacted_from_products = (
    changed_products.alias("p")
    .join(silver_orders_current.alias("o"), F.col("p.product_id") == F.col("o.product_id"))
    .select(F.col("o.order_id"))
    .distinct()
)

# Combine all impacted order IDs from orders, payments, and products, and remove duplicates
impacted_order_ids = (
    impacted_from_orders
    .union(impacted_from_payments)
    .union(impacted_from_products)
    .distinct()
)

# Print the DataFrame containing impacted order IDs
print("impacted_order_ids= ", impacted_order_ids)

# Display the impacted order IDs sorted by order_id
display(impacted_order_ids.orderBy("order_id"))

# **Step - 6 Build Gold delta for impacted orders**

This cell joins the impacted orders with the current Silver products and payment tables, derives business columns and builds Gold delta that will be merged into the Gold current-state table.

In [0]:
# Select distinct order IDs from changed orders
impacted_from_orders = changed_orders.select("order_id").distinct()  # Get unique order IDs from changed orders

# Select distinct order IDs from changed payments
impacted_from_payments = changed_payments.select("order_id").distinct()  # Get unique order IDs from changed payments

# Find order IDs impacted by changed products by joining with current orders on product_id
impacted_from_products = (
    changed_products.alias("p")  # Alias changed products as 'p'
    .join(silver_orders_current.alias("o"), F.col("p.product_id") == F.col("o.product_id"))  # Join with current orders on product_id
    .select(F.col("o.order_id"))  # Select order_id from orders
    .distinct()  # Get unique order IDs
)

# Combine all impacted order IDs from orders, payments, and products, and remove duplicates
impacted_order_ids = (
    impacted_from_orders  # Start with order IDs from changed orders
    .union(impacted_from_payments)  # Add order IDs from changed payments
    .union(impacted_from_products)  # Add order IDs from changed products
    .distinct()  # Remove duplicate order IDs
)

# Print the DataFrame containing impacted order IDs
print("impacted_order_ids= ", impacted_order_ids)  # Print the DataFrame object (not the data)

# Display the impacted order IDs sorted by order_id
display(impacted_order_ids.orderBy("order_id"))  # Show impacted order IDs in sorted order

In [0]:
# Join impacted orders with current orders, products, and payments to build the Gold delta DataFrame
gold_delta = (
    impacted_order_ids.alias("imp")  # Alias impacted order IDs as 'imp'
    .join(silver_orders_current.alias("o"), F.col("imp.order_id") == F.col("o.order_id"), "inner")  # Join with current orders on order_id
    .join(silver_product_current.alias("p"), F.col("o.product_id") == F.col("p.product_id"), "inner")  # Join with current products on product_id
    .join(silver_payment_current.alias("pay"), F.col("o.order_id") == F.col("pay.order_id"), "inner")  # Join with current payments on order_id
    .select(
        F.col("o.order_id"),  # Select order_id from orders
        F.col("o.customer_id"),  # Select customer_id from orders
        F.col("p.product_id"),  # Select product_id from products
        F.col("p.product_name"),  # Select product_name from products
        F.col("p.category"),  # Select category from products
        F.col("p.price").alias("product_price"),  # Select price from products as product_price
        F.col("o.order_status"),  # Select order_status from orders
        F.col("o.order_amount"),  # Select order_amount from orders
        F.col("pay.payment_id"),  # Select payment_id from payments
        F.col("pay.payment_status"),  # Select payment_status from payments
        F.col("pay.payment_date"),  # Select payment_date from payments
        F.lit("Credit card").cast("string").alias("payment_method"),  # Set payment_method as 'Credit card'
        F.col("pay.paid_amount").alias("payment_amount"),  # Select paid_amount from payments as payment_amount
        F.col("pay.paid_amount"),  # Select paid_amount from payments
        F.col("o.order_year").alias("order_year"),  # Select order_year from orders
        F.col("o.order_month").alias("order_month"),  # Select order_month from orders
        F.col("o.order_date").alias("order_date_only"),  # Select order_date from orders as order_date_only
        F.greatest(
            F.col("o.updated_at").cast("timestamp"),
            F.col("p.updated_at").cast("timestamp"),
            F.col("pay.processed_at").cast("timestamp")
        ).alias("gold_update_ts")  # Compute the latest update timestamp across orders, products, and payments
    )
    .dropDuplicates(["order_id"])  # Remove duplicate rows based on order_id
    .withColumn(
        "payment_completion_ratio",
        F.round(
            F.when(
                (F.col("order_amount") > 0) & (F.col("payment_amount") > 0),
                F.col("payment_amount") / F.col("order_amount")
            ).otherwise(F.lit(0.0)), 2
        )
    )  # Calculate payment completion ratio rounded to 2 decimals
    .withColumn(
        "payment_state",
        F.when(F.col("order_amount") == 0, "Invalid_order_amount")
         .when(F.col("payment_completion_ratio") == 0, "unpaid")
         .when(F.col("payment_completion_ratio") == 1, "paid")
         .when(F.col("payment_completion_ratio") < 1, "partially_paid")
         .when(F.col("payment_completion_ratio") > 1, "overpaid")
    )  # Derive payment state based on payment completion ratio
    .withColumn("gold_updated_date", F.to_date(F.col("gold_update_ts")))  # Extract date from gold_update_ts
    .withColumn("gold_run_id", F.lit(gold_run_id))  # Add gold_run_id for tracking
)
print("gold_delta_rows= ", gold_delta.count())  # Print the number of rows in gold_delta
display(gold_delta.limit(5))  # Display the first 5 rows of gold_delta

### **Step 7 - Merge Gold current-state table**

If Gold delta contains rows, this cell merges them into **gold_schema.orders_information.**

If there are no impacted rows, nothing is merged

In [0]:
# Step 7 - Merge Gold current-state table
# If there are new or changed gold_delta rows, merge them into the Gold orders_information table
if gold_delta.count() > 0:
    upsert_to_gold(
        gold_delta,  # DataFrame containing new/changed Gold rows
        "databricks_dev.`02_gold`.orders_information",  # Target Gold table
        "order_id"  # Join key for upsert
    )
else:
    # If no impacted rows, nothing is merged
    print("no new data to write to gold")

In [0]:
%sql
select * from databricks_dev.`02_gold`.orders_information

### **Step 8 - Maintain gold SCD Type 2 history**
This cell updates SCD2 history table.

If acurrent Gold row changes, the old version is closed(**is_current = false)** and a new current version is inserted.

In [0]:
# Check if the SCD2 history table exists; if not, create it with the required schema and columns
if not spark.catalog.tableExists("databricks_dev.`02_gold`.orders_information_scd2"):
    spark.sql("""
        create table databricks_dev.`02_gold`.orders_information_scd2 using delta as
        select *,
               cast(null as timestamp) as valid_from_ts,
               cast(null as timestamp) as valid_to_ts,
               true as is_current
        from databricks_dev.`02_gold`.orders_information
        where 1 = 0
    """)

# If there are new or changed gold_delta rows, update SCD2 history
if gold_delta.count() > 0:
    # Register gold_delta as a temp view for SQL operations
    gold_delta.createOrReplaceTempView("gold_delta_view")
    
    # Mark previous current records as not current and set their valid_to_ts if any tracked columns changed
    spark.sql("""
        merge into databricks_dev.`02_gold`.orders_information_scd2 t
        using gold_delta_view s
        on t.order_id = s.order_id and t.is_current = true
        when matched and (
            not(t.order_status <=> s.order_status) or
            not(t.order_amount <=> s.order_amount) or
            not(t.paid_amount <=> s.paid_amount) or
            not(t.payment_id <=> s.payment_id) or
            not(t.category <=> s.category) or
            not(t.product_name <=> s.product_name) or
            not(t.product_price <=> s.product_price)
        )
        then update set
            t.valid_to_ts = s.gold_update_ts,
            t.is_current = false
    """)

    # Insert new current records for changed or new order_ids
    spark.sql("""
        insert into databricks_dev.`02_gold`.orders_information_scd2
        select s.*, 
               s.gold_update_ts as valid_from_ts,
               cast(null as timestamp) as valid_to_ts,
               true as is_current
        from gold_delta_view s
        left join databricks_dev.`02_gold`.orders_information_scd2 t
            on s.order_id = t.order_id and t.is_current = true
        where t.order_id is null or (
            not(t.order_status <=> s.order_status) or
            not(t.order_amount <=> s.order_amount) or
            not(t.paid_amount <=> s.paid_amount) or
            not(t.payment_id <=> s.payment_id) or
            not(t.category <=> s.category) or
            not(t.product_name <=> s.product_name) or
            not(t.product_price <=> s.product_price)
        )
    """)

### **Step 9 - Update category-level Gold aggregation**

This cell recalculates catagory level business metrics only for categories  impacted in the current run, then merges them into the category performance Gold table.

In [0]:
if gold_delta.count() > 0:
    # Extract distinct impacted categories from gold_delta
    impacted_categories = (
        gold_delta
        .select("category")  # Select the category column
        .filter(F.col("category").isNotNull())  # Filter out null categories
        .distinct()  # Get unique categories
    )
    # Aggregate category-level metrics for only impacted categories
    category_performance_delta = (
        spark.read.table("databricks_dev.`02_gold`.orders_information")  # Read the Gold orders table
        .join(impacted_categories, on="category", how="inner")  # Join with impacted categories
        .groupBy("category")  # Group by category
        .agg(
            F.sum(
                F.when(F.col("order_amount") > 0, F.col("order_amount")).otherwise(F.lit(0.0))
            ).alias("Gross_merchandise_Value"),  # Sum of positive order_amounts
            F.sum(
                F.when(F.col("paid_amount") > 0, F.col("paid_amount")).otherwise(F.lit(0.0))
            ).alias("total_paid_amount"),  # Sum of positive paid_amounts
            F.countDistinct(F.col("order_id")).alias("total_orders"),  # Count of unique orders
            F.avg(F.col("payment_completion_ratio")).alias("avg_payment_completion_ratio"),  # Average payment completion ratio
            (
                F.sum(F.when(F.col("payment_status") == "FAILED", 1).otherwise(F.lit(0))) /
                F.count(F.lit(1))
            ).alias("total_failed_payments_rate")  # Rate of failed payments
        )
    )
    # Merge the aggregated metrics into the Gold category performance table
    upsert_to_gold(category_performance_delta, "databricks_dev.`02_gold`.category_performance", "category")
display(category_performance_delta)

In [0]:
%sql

select * from databricks_dev.`02_gold`.category_performance


### **Step 10 - Publish Gold snapshots to Volume**

This cell writes two kind of Gold outputs to a Databricks Volume:

- **Latest snapshot -**> Overwritten evry successful run
- **timestamped historical snapshot** -> a new folder for each successful run

This is useful for audit, rollback and teaching demos.

In [0]:
spark.sql("create volume if not exists databricks_dev.`02_gold`.gold_snapshots_vol")

In [0]:
# Define the Volume paths for Gold snapshots
latest_orders_path = ("/Volumes/databricks_dev/02_gold/gold_snapshots_vol/gold_latest/orders_information")  # Path for latest orders_information snapshot
latest_category_path = ("/Volumes/databricks_dev/02_gold/gold_snapshots_vol/gold_latest/category_performance")  # Path for latest category_performance snapshot

historical_orders_path = f"/Volumes/databricks_dev/02_gold/gold_snapshots_vol/gold_snapshots/orders_information/run_date={run_date_str}/run_ts={run_ts_str}"  # Path for timestamped historical orders_information snapshot
historical_category_path = f"/Volumes/databricks_dev/02_gold/gold_snapshots_vol/gold_snapshots/category_performance/run_date={run_date_str}/run_ts={run_ts_str}"  # Path for timestamped historical category_performance snapshot

# Write the latest Gold orders_information snapshot to the Volume (overwrites previous)
spark.read.table("databricks_dev.`02_gold`.orders_information").write.mode("overwrite").format("parquet").save(latest_orders_path)

# Write the latest Gold category_performance snapshot to the Volume (overwrites previous)
spark.read.table("databricks_dev.`02_gold`.category_performance").write.mode("overwrite").format("parquet").save(latest_category_path)

# Write a timestamped historical Gold orders_information snapshot to the Volume
spark.read.table("databricks_dev.`02_gold`.orders_information").write.mode("overwrite").format("parquet").save(historical_orders_path)

# Write a timestamped historical Gold category_performance snapshot to the Volume
spark.read.table("databricks_dev.`02_gold`.category_performance").write.mode("overwrite").format("parquet").save(historical_category_path)

# Print the snapshot paths for reference
print("latest_orders_path: ", latest_orders_path)
print("latest_category_path: ", latest_category_path)
print("historical_orders_path: ", historical_orders_path)
print("historical_category_path: ", historical_category_path)


**Step 11 - update Gold control table**

This final cell updates the gold control table using the latest Silver processing metadata and displays the control table for validation


In [0]:
# Get the latest bronze_ingested_ts from the current Silver orders
latest_silver_ts = silver_orders_current.agg(F.max("bronze_ingested_ts").alias("mx")).collect()[0]["mx"]

# Get the latest silver_run_id for the latest bronze_ingested_ts
latest_silver_run_id = (
    silver_orders_current
    .filter(F.col("bronze_ingested_ts") == latest_silver_ts)
    .agg(F.max("silver_run_id").alias("mx"))
    .collect()[0]["mx"]
) if latest_silver_ts is not None else None

# Upsert the Gold control table with the latest run metadata
upsert_gold_control("orders_information", latest_silver_run_id, latest_silver_ts, gold_delta.count())

# Display the updated Gold processing control table
display(spark.table("databricks_dev.`02_gold`.processing_control"))

In [0]:
%sql
-- One-time fix: rename the stale 'order_information' row to 'orders_information'
-- so the existing watermark (last_processed_silver_run_ts) is preserved and used on the next run
UPDATE databricks_dev.`02_gold`.processing_control
SET entity_name = 'orders_information'
WHERE entity_name = 'order_information'

In [0]:
%sql
select * from databricks_dev.`02_gold`.processing_control

In [0]:
%sql
select * from databricks_dev.`02_gold`.orders_information_scd2 where order_id=200002